In [ ]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import string
import random
import spacy as sp
import zipfile
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
path = "/content/archive (3).zip"
# Instantiating the ZipFile class
zip_obj = zipfile.ZipFile(path, "r")
# Extracting the files into the project root
zip_obj.extractall("./")
zip_obj.close()

In [ ]:
# path to fake news data
path_fake_data = "/content/Fake.csv"
fake_news = pd.read_csv(path_fake_data)
fake_news.head()

In [ ]:
fake_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [ ]:
fake_news.duplicated().sum()

3

In [ ]:
fake_news.drop_duplicates(inplace=True)

In [ ]:
fake_news["target"] = 0
fake_news

In [ ]:
print(f"There are {len(fake_news)} texts\n")
print(f"The first 2 texts are these:\n")
for i in range(2):
  print(fake_news["text"][i])

There are 23478 texts

The first 2 texts are these:

Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America!  Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting like this despicable, petty, infantile

In [ ]:
unics, counts = np.unique(fake_news['subject'], return_counts=True)
unics, counts

(array(['Government News', 'Middle-east', 'News', 'US_News', 'left-news',
        'politics'], dtype=object),
 array([1570,  778, 9050,  783, 4459, 6838]))

In [ ]:
# path to real news data
path_true_data = "/content/True.csv"
true_news = pd.read_csv(path_true_data)
true_news.head()

In [ ]:
true_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [ ]:
true_news.duplicated().sum()

206

In [ ]:
true_news.drop_duplicates(inplace=True)

In [ ]:
true_news["target"] = 1
true_news.head()

In [ ]:
print(f"There are {len(true_news)} texts\n")
print(f"The first 2 texts are these:\n")
for i in range(2):
  print(true_news["text"][i])

There are 21211 texts

The first 2 texts are these:

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they return from the holidays on Wednesday, lawmakers will begin trying to pass a federal budget in a fight likely to be linked to other issues, such as immigration policy, even as the November congressional election campaigns approach in which Republicans will seek to keep control of Congress. President Donald Trump and his Republicans want a big budget increase in military spending, while Democrats also want proportional increases for non-defense “dis

In [ ]:
unics, counts = np.unique(true_news['subject'], return_counts=True)
unics, counts

(array(['politicsNews', 'worldnews'], dtype=object), array([11220,  9991]))

In [ ]:
true_news.drop(["subject", "date", "title"], axis=1, inplace=True)
fake_news.drop(["subject", "date", "title"], axis=1, inplace=True)

In [ ]:
# Equaling the amount of news in two dataframes

total_news = 2000
fake_news = fake_news[:total_news]
true_news = true_news[:total_news]
len(fake_news), len(true_news)

(2000, 2000)

In [ ]:
news = pd.concat([true_news, fake_news], ignore_index=True)
news = news.sample(frac=1, random_state=42).reset_index(drop=True)
news.head()

In [ ]:
unics, counts = np.unique(news['target'], return_counts=True)
unics, counts

(array([0, 1]), array([2000, 2000]))

In [ ]:
print(f"There are {len(news)} texts\n")
print(f"The first 2 texts are these:\n")
for i in range(2):
  print(news["text"][i], news["target"][i])

There are 4000 texts

The first 2 texts are these:

NEW YORK (Reuters) - Haitian immigrants on Tuesday decried a U.S. decision to end a program that granted 59,000 Haitians temporary visas after the 2010 earthquake, saying they would be sent back to a country that has yet to recover from that disaster and others since. The United States offered Temporary Protected Status, or TPS, to Haitians after the January 2010 earthquake killed some 300,000 people and devastated a country that has long been the poorest in the Americas. The administration of former President Barack Obama  extended the program several times, finding that conditions in Haiti were too dire to send the beneficiaries home. President Donald Trump’s administration, after previously granting a six-month extension, announced on Monday that it would end TPS for Haiti in July 2019. Any Haitian who cannot obtain another kind of U.S. visa will be subject to deportation back to the Caribbean nation, where some earthquake victims 

In [ ]:
nlp = sp.load("en_core_web_sm")
stop_words = sp.lang.en.STOP_WORDS

In [ ]:
def pre_processing(text):
    """

    Args:
      text: texts (str type) with stop words, unwanted characters and punctuations

    Returns:
      clean texts without stop words and punctuation

    """
    text = BeautifulSoup(text, "lxml").get_text()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text).strip()
    text = re.sub(r"  +", " ", text).strip()
    text = text.lower()
    doc = nlp(text)
    words = [token.text for token in doc if not token.is_stop and token.text not in string.punctuation]
    text = " ".join(words)

    return text

In [ ]:
txt = news["text"][1]
txt

'Remember when Republicans were told to stop talking about rape because they were repeatedly sticking their foot in their mouths? Well, now they are being told to stop talking about their attempt to repeal the Affordable Care Act.Ever since Republicans giddily started preparing to repeal the landmark healthcare law, they faced a sea of angry constituents at town hall events across the country. As the GOP dug in their heels to strip healthcare from millions of Americans, their own voting base began to realize that their own healthcare was threatened as well and that the law they referred to as Obamacare wasn t so bad after all.In fact, many realize that the law saved their lives and they don t want Republicans to repeal it and put their lives back in jeopardy.But Republicans shoved their bill through the House without even reading it and getting the CBO to score it. The bill lets states opt out of requiring insurance companies to cover people with pre-existing conditions, which have now

In [ ]:
print(pre_processing(txt))

remember republicans told stop talking rape repeatedly sticking foot mouths told stop talking attempt repeal affordable care act republicans giddily started preparing repeal landmark healthcare law faced sea angry constituents town hall events country gop dug heels strip healthcare millions americans voting base began realize healthcare threatened law referred obamacare wasn t bad fact realize law saved lives don t want republicans repeal lives jeopardy republicans shoved bill house reading getting cbo score bill lets states opt requiring insurance companies cover people pre existing conditions defined including sexual assault pregnancy cancer acne asthma diabetes sleep apnea alzheimer s disease host conditions medical problems list find pretty allow insurance companies discriminate millions americans conservatives included bill slashed billion medicaid covers millions low income americans order fund tax gift wealthy millions americans including conservatives red states got healthcare 

In [ ]:
sentences = list(news["text"])
labels = list(news["target"])

In [ ]:
sentences = [pre_processing(sentence) for sentence in sentences]

In [ ]:
for _ in range(2):
  print(sentences[random.randint(0, len(news) -1 )])

washington reuters u s house representatives wednesday approved roughly billion initial emergency aid relief rebuilding hurricane harvey tore texas aug house passed measure provides billion federal emergency management agency million small business administration senate barring unexpected setbacks aid measure expected sent white house end week white house meeting president donald trump congressional leaders parties senate minority leader chuck schumer house minority leader nancy pelosi announced reached deal tie harvey aid short term measures fund government raise debt ceiling dec sides intention avoiding default december look forward working issues schumer pelosi said joint statement senate majority leader mitch mcconnell favored linking harvey aid measure raise debt ceiling federal government s borrowing limit midterm elections aides said conservative republicans oppose tying harvey aid debt ceiling measure related reforms republicans white house meeting pressed raise debt ceiling lo

In [ ]:
TRUNC='post'
PADDING = 'post'
OOV_TOKEN = "<OOV>"
EMBEDDING = 120
EPOCHS = 15
TRAINING_SPLIT = 0.8
maxlen = max([len(sentence.split()) for sentence in sentences])

In [ ]:
def train_val_split(sentences, labels, training_split):
    """
    Splits the dataset into training and validation sets

    Args:
        sentences (list of string): lower-cased sentences without stopwords
        labels (list of string): list of labels
        training split (float): proportion of the dataset to convert to include in the train set

    Returns:
        train_sentences, validation_sentences, train_labels, validation_labels - lists containing the data splits
    """

    # Compute the number of sentences that will be used for training (should be an integer)
    train_size = int(len(sentences) * training_split)

    # Split the sentences and labels into train/validation splits
    train_sentences = sentences[:train_size]
    train_labels = labels[:train_size]

    validation_sentences = sentences[train_size:]
    validation_labels = labels[train_size:]

    return train_sentences, validation_sentences, train_labels, validation_labels

In [ ]:
train_news, val_news, train_labels, val_labels = train_val_split(sentences, labels, TRAINING_SPLIT)

print(f"There are {len(train_news)} sentences for training.\n")
print(f"There are {len(train_labels)} labels for training.\n")
print(f"There are {len(val_news)} sentences for validation.\n")
print(f"There are {len(val_labels)} labels for validation.")

There are 3200 sentences for training.

There are 3200 labels for training.

There are 800 sentences for validation.

There are 800 labels for validation.


In [ ]:
def fit_tokenizer(train_sentences, oov_token):
    """
    Instantiates the Tokenizer class on the training sentences

    Args:
        train_sentences (list of string): lower-cased sentences without stopwords to be used for training
        oov_token (string) - symbol for the out-of-vocabulary token

    Returns:
        tokenizer (object): an instance of the Tokenizer class containing the word-index dictionary
    """

    # Instantiate the Tokenizer class, passing in the correct values for oov_token
    tokenizer = Tokenizer(oov_token=oov_token)

    # Fit the tokenizer to the training sentences
    tokenizer.fit_on_texts(train_sentences)

    return tokenizer

In [ ]:
tokenizer = fit_tokenizer(sentences, OOV_TOKEN)
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index)

In [ ]:
def seq_pad_and_trunc(sentences, tokenizer, padding, truncating, maxlen):
    """
    Generates an array of token sequences and pads them to the same length

    Args:
        sentences (list of string): list of sentences to tokenize and pad
        tokenizer (object): Tokenizer instance containing the word-index dictionary
        padding (string): type of padding to use
        truncating (string): type of truncating to use
        maxlen (int): maximum length of the token sequence

    Returns:
        pad_trunc_sequences (array of int): tokenized sentences padded to the same length
    """
    # Convert sentences to sequences
    sequences = tokenizer.texts_to_sequences(sentences)

    # Pad the sequences using the correct padding, truncating and maxlen
    pad_trunc_sequences = pad_sequences(sequences, padding=padding, truncating=truncating, maxlen=maxlen)

    return pad_trunc_sequences

In [ ]:
train_pad_trunc_seq = seq_pad_and_trunc(train_news, tokenizer, PADDING, TRUNC, maxlen)
val_pad_trunc_seq = seq_pad_and_trunc(val_news, tokenizer, PADDING, TRUNC, maxlen)

print(f"Padded and truncated training sequences have shape: {train_pad_trunc_seq.shape}\n")
print(f"Padded and truncated validation sequences have shape: {val_pad_trunc_seq.shape}")

Padded and truncated training sequences have shape: (3200, 791)

Padded and truncated validation sequences have shape: (800, 791)


In [ ]:
for _ in range(2):
  print(val_pad_trunc_seq[random.randint(0, len(val_pad_trunc_seq) -1 )])

[   14  1813   122   809   622  1376     9     3  2727   620    48    99
   125    90  3135   715     3    89   370   629    56  1112     2  1328
   223     2   515   148   370   629    56    72    18    68 30573   809
   622    88     3    85  4418    89   622     4   290     7   305    14
  1813   122   583   135     7    22   219   377  2586    12     8  1232
 13598  1369    35  3428  5965    85     3   134  1009  2727     3     2
   262  5766   622   176   273   388     2  1178     7   118   737 10383
   253   498  1298     2  4118   278    11   134    55     2   278   479
   293   479   135     7    22   219     3  5423   560   638   746  7252
  2176   622   135     7    22   219 13356    64  3327    83  2447  1116
  2150   837    50     3     2   367   244     3     2   434   268  1216
   622    37    31   396  3453   635    47    45     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

In [ ]:
train_labels = np.array(train_labels)
val_labels = np.array(val_labels)

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs={}):
            if logs.get('val_accuracy') is not None and logs.get('val_accuracy') > 0.995:
                print("\nReached 99,5% val_accuracy so cancelling training!")
                # Stop training when the above condition is met
                self.model.stop_training = True

In [ ]:
def create_model(vocab_size, embedding_dim, maxlen):
    """
    Creates a binary sentiment classifier model

    Args:
        vocab_size (int): size of the vocabulary for the Embedding layer input
        embedding_dim (int): dimensionality of the Embedding layer output
        maxlen (int): length of the input sequences
    Returns:
        model (tf.keras Model): the sentiment classifier model
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size+1, embedding_dim, input_length=maxlen),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

    return model

In [ ]:
model = create_model(VOCAB_SIZE, EMBEDDING, maxlen)
callbacks = myCallback()

# Train the model and save the training history
history = model.fit(
    train_pad_trunc_seq, train_labels, epochs=EPOCHS,
    validation_data=(val_pad_trunc_seq, val_labels),
    callbacks=[callbacks]
    )

Epoch 1/15
100/100 [==============================] - 8s 72ms/step - loss: 0.5632 - accuracy: 0.7028 - val_loss: 0.0699 - val_accuracy: 0.9887
Epoch 2/15
100/100 [==============================] - 6s 57ms/step - loss: 0.0378 - accuracy: 0.9881 - val_loss: 0.0260 - val_accuracy: 0.9912
Epoch 3/15
100/100 [==============================] - 6s 65ms/step - loss: 0.0053 - accuracy: 0.9991 - val_loss: 0.0174 - val_accuracy: 0.9937
Epoch 4/15
100/100 [==============================] - 6s 63ms/step - loss: 0.0014 - accuracy: 0.9997 - val_loss: 0.0263 - val_accuracy: 0.9925
Epoch 5/15
100/100 [==============================] - 6s 57ms/step - loss: 5.2233e-04 - accuracy: 1.0000 - val_loss: 0.0243 - val_accuracy: 0.9925
Epoch 6/15
100/100 [==============================] - 7s 70ms/step - loss: 2.2504e-04 - accuracy: 1.0000 - val_loss: 0.0292 - val_accuracy: 0.9925
Epoch 7/15
100/100 [==============================] - 6s 58ms/step - loss: 2.0309e-04 - accuracy: 1.0000 - val_loss: 0.0356 - val_accu

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.plot(acc)
plt.plot(val_acc)
plt.title('Training and validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training', 'Validation'], loc='lower right')
plt.show()

plt.plot(loss)
plt.plot(val_loss)
plt.title('Training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training', 'Validation'], loc='upper right')
plt.show()